# MUFASA — continued pretraining: Qwen3.5 0.8B Base

Production rsLoRA CPT over the licence-filtered MUFASA paper corpus.

Key safety properties:

- papers are cleaned once under one versioned policy;
- every source token is retained in pre-tokenized 4,096-token windows;
- exactly one EOS is appended at the real end of each paper, never at artificial cuts;
- a real two-step optimizer preflight checks memory, finite gradients, and adapter updates;
- held-out loss selects the best checkpoint;
- resume is allowed only when the model, corpus, tokenizer, packages, and training contract match;
- evaluation uses the same cleaning policy and deterministic held-out samples.

This is the text-only `unsloth/Qwen3.5-0.8B-Base` checkpoint, loaded with Unsloth `FastModel`. It is not the post-trained or vision model. The adapter contract covers all 18 Gated-DeltaNet layers, all 6 full-attention layers, every MLP, and the embedding/output matrices.

The 4,096-token context is a deliberate efficiency choice, independent of GPU tier.
Lossless consecutive windows preserve the full paper, while batch 2 × accumulation 16
preserves the intended effective token batch. Use a new `RUN_NAME` if this changes.


In [ ]:
# ============================== environment setup ============================
# Run first. Checkpoints are durable in Colab Drive; corpus reads stay local.
import os
import sys
from pathlib import Path

RUN_SCHEMA = "mufasa-rslora-cpt-v2"
RUN_NAME = "qwen35-0.8b-base-cpt-v2"  # v2 deliberately cannot resume flawed v1 checkpoints
DRIVE_DIR = "mufasa"
CORPUS_DIR = ""  # optional exact path, e.g. /content/drive/MyDrive/mufasa/corpus_splits

# Deliberate efficiency choice, independent of hardware tier. Lossless windows
# preserve every eligible paper token and the real preflight measures VRAM.
MAX_SEQ_LENGTH = 4096
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 16
PREFLIGHT_STEPS = 2
SAVE_STEPS = 100
SEED = 3407

try:
    from google.colab import drive as _colab_drive
    HAS_COLAB_FRONTEND = True
except ImportError:
    _colab_drive = None
    HAS_COLAB_FRONTEND = False
ON_COLAB_VM = Path("/content").is_dir()
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / DRIVE_DIR

if HAS_COLAB_FRONTEND:
    _colab_drive.mount("/content/drive")
    CKPT_DIR = DRIVE / "checkpoints" / RUN_NAME
elif ON_COLAB_VM:
    CKPT_DIR = Path("/content/checkpoints") / RUN_NAME
    print("WARNING: checkpoints are on the ephemeral VM disk; copy them off the VM.")
else:
    CKPT_DIR = Path("outputs") / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def _is_corpus(path):
    return path.is_dir() and all((path / split / "markdown").is_dir()
                                 for split in ("train", "evaluate", "test"))


def find_corpus():
    # Set CORPUS_DIR only if your Drive layout is different from these normal
    # locations. This notebook reads the mounted folder directly; no ZIP is used.
    candidates = []
    if CORPUS_DIR:
        candidates.append(Path(CORPUS_DIR).expanduser())
    candidates.extend([
        Path.cwd() / "corpus_splits",
        Path.cwd() / "01-data-engineering" / "data-extraction" / "corpus_splits",
        DRIVE / "corpus_splits",
        DRIVE / "01-data-engineering" / "data-extraction" / "corpus_splits",
        DRIVE_ROOT / "corpus_splits",
        DRIVE_ROOT / "MUFASA" / "corpus_splits",
        DRIVE_ROOT / "AfricanDeepTechChallenge" / "MUFASA" / "corpus_splits",
        DRIVE_ROOT / "AfricanDeepTechChallenge" / "corpus_splits",
    ])
    for candidate in candidates:
        if _is_corpus(candidate):
            return candidate
    attempted = "\n  - ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find the mounted corpus folder. It must contain "
        "train/markdown, evaluate/markdown, and test/markdown. "
        "Set CORPUS_DIR near the top of this cell to its exact Drive path."
        f"\nTried:\n  - {attempted}"
    )


MUFASA_ROOT = find_corpus()
sys.path.insert(0, str(MUFASA_ROOT))
sys.path.insert(0, str(Path.cwd()))

import torch
assert torch.cuda.is_available(), "Select a GPU runtime before loading the model"
card = torch.cuda.get_device_properties(0)
memory = card.total_memory / 1e9
print(f"GPU        : {card.name}  {memory:,.1f} GB")
print(f"precision  : {'bfloat16' if torch.cuda.is_bf16_supported() else 'model-safe fallback'}")
print(f"window     : {MAX_SEQ_LENGTH:,} tokens; batch={TRAIN_BATCH_SIZE}; accum={GRADIENT_ACCUMULATION_STEPS}")
for split in ("train", "evaluate", "test"):
    found = MUFASA_ROOT / split / "markdown"
    print(f"{split:<11}: {len(list(found.glob('*.md'))):,} papers")
print(f"corpus     : {MUFASA_ROOT}")
print(f"checkpoints: {CKPT_DIR}")


Run this on a Colab CUDA GPU of your choice. The notebook does not assume a
specific GPU tier: its two-step optimizer preflight measures the selected device before
the full run begins. Package versions, corpus identity, and training settings are pinned
in the run manifest.


### News


Introducing **[Unsloth Desktop](https://unsloth.ai/docs/desktop)**, the first desktop app to run and train models. Free and open-source for macOS, Windows and Linux. [GitHub](https://github.com/unslothai/unsloth) • [Download](https://unsloth.ai/download)

<p>
<a href="https://unsloth.ai/docs/desktop"><img src="https://raw.githubusercontent.com/unslothai/notebooks/refs/heads/main/assets/unsloth-qwen3-8.png" width="350" alt="Introducing Unsloth Desktop"></a>
</p>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).


### Installation


In [ ]:
%%capture
# A coherent, current stack. Qwen3.5 requires Transformers v5; the same pinned
# stack is used for all three candidates so their runs are reproducible.
%pip install -qU "unsloth[colab-new,colab-no-deps]==2026.8.19" "unsloth_zoo==2026.8.13" "transformers==5.5.0" "trl==0.24.0" "datasets==4.3.0"


### Unsloth


In [ ]:
# Keep Unsloth's low-memory fused causal loss enabled. The old `%env ... #`
# line accidentally embedded its comment in the value; this is explicit.
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "0"
assert os.environ["UNSLOTH_RETURN_LOGITS"] == "0"
print("UNSLOTH_RETURN_LOGITS =", repr(os.environ["UNSLOTH_RETURN_LOGITS"]))


In [ ]:
from importlib.metadata import version

import torch
from unsloth import FastModel

MODEL_API = FastModel
MODEL_ID = "unsloth/Qwen3.5-0.8B-Base"
max_seq_length = MAX_SEQ_LENGTH
dtype = None
load_in_4bit = False
USE_BF16 = torch.cuda.is_bf16_supported()
# Qwen3.5 Gated-DeltaNet is explicitly protected from FP16 gradient NaNs by
# current Unsloth. On a GPU without BF16, load float32 explicitly rather than
# relying on a hardware-specific implicit dtype.
USE_FP16 = False
dtype = torch.bfloat16 if USE_BF16 else torch.float32


PACKAGE_VERSIONS = {
    name: version(name)
    for name in ("unsloth", "unsloth_zoo", "transformers", "trl", "datasets", "peft", "bitsandbytes")
}
PACKAGE_VERSIONS["torch"] = torch.__version__
print("versions:", PACKAGE_VERSIONS)

from huggingface_hub import model_info

BASE_COMMIT = model_info(MODEL_ID).sha
if not BASE_COMMIT:
    raise RuntimeError(f"Could not resolve an immutable Hub revision for {MODEL_ID}")

model, tokenizer = MODEL_API.from_pretrained(
    model_name = MODEL_ID,
    revision = BASE_COMMIT,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    text_only = True,
    use_gradient_checkpointing = "unsloth",

)
if tokenizer.eos_token_id is None:
    raise RuntimeError("The tokenizer has no EOS token; paper boundaries cannot be represented")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

LOADED_COMMIT = (
    getattr(model.config, "_commit_hash", None)
    or getattr(getattr(model.config, "text_config", None), "_commit_hash", None)
    or tokenizer.init_kwargs.get("_commit_hash")
)
if LOADED_COMMIT and LOADED_COMMIT != BASE_COMMIT:
    raise RuntimeError(f"loaded revision {LOADED_COMMIT} != requested {BASE_COMMIT}")
print("model       :", MODEL_ID)
print("base commit :", BASE_COMMIT)
print("precision   :", "bf16" if USE_BF16 else ("fp16" if USE_FP16 else "model-safe fp32"))

loss_fn = getattr(model, "loss_function", None)
loss_name = f"{getattr(loss_fn, '__module__', '')}.{getattr(loss_fn, '__name__', repr(loss_fn))}"
assert "unsloth" in loss_name.casefold(), f"low-memory Qwen loss was not installed: {loss_name}"
print("loss backend:", loss_name)


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

We also add `embed_tokens` and `lm_head` to allow the model to learn out of distribution data.


In [ ]:
from collections import Counter
import re

from peft.tuners.lora.layer import LoraLayer
from peft.utils.other import ModulesToSaveWrapper

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
    "in_proj_qkv",
    "in_proj_z",
    "in_proj_a",
    "in_proj_b",
    "out_proj",
    "embed_tokens",
    "lm_head"
]
LORA_R = 128
LORA_ALPHA = 32
EXPECTED_MAIN = Counter({
    "q_proj": 6, "k_proj": 6, "v_proj": 6, "o_proj": 6,
    "gate_proj": 24, "up_proj": 24, "down_proj": 24,
    "in_proj_qkv": 18, "in_proj_z": 18, "in_proj_a": 18,
    "in_proj_b": 18, "out_proj": 18,
})

def _main_layer_id(path):
    dotted = "." + path
    if ".mtp." in dotted:
        return None
    match = re.search(r"(?:^|\.)(?:language_model\.)?layers\.(\d+)\.", path)
    return int(match.group(1)) if match else None

cfg = getattr(model.config, "text_config", model.config)
layer_types = list(cfg.layer_types)
assert len(layer_types) == 24
assert Counter(layer_types) == Counter({"linear_attention": 18, "full_attention": 6})

_before_paths = {
    name for name, _ in model.named_modules()
    if name.rsplit(".", 1)[-1] in TARGET_MODULES
}
_before_main = Counter(
    name.rsplit(".", 1)[-1] for name in _before_paths
    if _main_layer_id(name) is not None
)
assert _before_main == EXPECTED_MAIN, (_before_main, EXPECTED_MAIN)
_before_all = Counter(name.rsplit(".", 1)[-1] for name in _before_paths)
assert _before_all["embed_tokens"] == 1
assert _before_all["lm_head"] == 1

model = MODEL_API.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = True,
    loftq_config = None,
)

_actual_paths = {name for name, module in model.named_modules() if isinstance(module, LoraLayer)}
_full_paths = {name for name, module in model.named_modules()
               if isinstance(module, ModulesToSaveWrapper)}

def _covers(actual, expected):
    return actual == expected or actual.endswith("." + expected)

_special = {"embed_tokens", "lm_head"}
_projection_paths = {p for p in _before_paths if p.rsplit(".", 1)[-1] not in _special}
_missed = sorted(path for path in _projection_paths
                 if not any(_covers(a, path) for a in _actual_paths))
_special_missed = sorted(
    path for path in _before_paths if path.rsplit(".", 1)[-1] in _special
    and not any(_covers(a, path) for a in (_actual_paths | _full_paths))
)
_unexpected = sorted(path for path in _actual_paths | _full_paths
                     if not any(_covers(path, e) for e in _before_paths))
assert not _missed, f"LoRA targets missed: {_missed[:20]}"
assert not _special_missed, f"embedding/head targets missed: {_special_missed}"
assert not _unexpected, f"unexpected LoRA targets: {_unexpected[:20]}"

_after_main = Counter(
    name.rsplit(".", 1)[-1] for name in _actual_paths
    if _main_layer_id(name) is not None
)
assert _after_main == EXPECTED_MAIN, (_after_main, EXPECTED_MAIN)
assert {_main_layer_id(n) for n in _actual_paths if _main_layer_id(n) is not None} == set(range(24))
for path in _before_paths:
    if path.rsplit(".", 1)[-1] in _special:
        matches = sum(_covers(a, path) for a in (_actual_paths | _full_paths))
        assert matches == 1, (path, matches)

ADAPTER_CONTRACT = dict(sorted(_before_all.items()))
ADAPTER_MODES = {
    target: ("modules_to_save" if any(p.endswith("." + target) or p == target for p in _full_paths)
             else "lora")
    for target in TARGET_MODULES
}
print(f"adapter contract OK: {len(_actual_paths)} LoRA + {len(_full_paths)} full wrappers; all 24 text layers covered")
print("main-layer targets:", dict(sorted(_after_main.items())))
print("embedding/head modes:", {k: ADAPTER_MODES[k] for k in _special})


<a name="Data"></a>
### Lossless MUFASA paper windows

The corpus is supplied directly as pre-tokenized causal-LM examples. No Wikipedia
template, translation, or artificial document endings are used.


In [ ]:
# No formatting function is used. The dataset cell below supplies exact token IDs,
# and the causal collator constructs labels while masking padding by position.


The complete eligible training split is used for one epoch; no percentage subsample is taken.


In [ ]:
# ======================== lossless MUFASA dataset ===========================
# Pretokenized rows are intentional: TRL otherwise re-tokenizes text and may
# append EOS to every artificial window. Here only a real paper end receives EOS.
import hashlib
import re
from pathlib import Path

import pyarrow.compute as pc
from datasets import Dataset, Features, Sequence, Value
from tqdm.auto import tqdm

CORPUS = MUFASA_ROOT / "train" / "markdown"
DROP_REFERENCES = True
DROP_FRONT_MATTER = True
MIN_CHARS = 2000
MIN_FINAL_WINDOW = 128
WINDOWING_VERSION = "mufasa-token-windows-v2"
CLEANER_VERSION = "mufasa-cpt-cleaner-v2"

FRONT_MATTER = re.compile(r"\A---\n.*?\n---\n", re.S)
PAGE_MARKER = re.compile(r"^<!--\s*MUFASA_PDF_PAGE.*?-->\s*$", re.M)
PAGE_HEADING = re.compile(r"^#{1,6}\s*PDF page \d+\s*$", re.M)
REFERENCES = re.compile(
    r"^\s*(?:#{1,6}\s*)?\|?\s*(?:\*\*|__)?\s*"
    r"(?:references?|bibliography|works cited|literature cited)"
    r"\s*(?:\*\*|__)?\s*\|?\s*$",
    re.I | re.M,
)


def clean_paper(text):
    """The one cleaning policy used for train, live eval, and final eval."""
    if DROP_FRONT_MATTER:
        text = FRONT_MATTER.sub("", text)
    text = PAGE_MARKER.sub("", text)
    text = PAGE_HEADING.sub("", text)
    if DROP_REFERENCES:
        headings = list(REFERENCES.finditer(text))
        if headings:
            text = text[:headings[-1].start()]
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def window_token_ids(token_ids, max_length, eos_token_id, min_final=MIN_FINAL_WINDOW):
    """Lossless ordered windows with exactly one authored EOS at paper end."""
    sequence = list(token_ids) + [int(eos_token_id)]
    chunks = [sequence[i:i + max_length] for i in range(0, len(sequence), max_length)]
    if len(chunks) > 1 and len(chunks[-1]) < min_final:
        needed = min_final - len(chunks[-1])
        assert len(chunks[-2]) - needed >= 2
        chunks[-1] = chunks[-2][-needed:] + chunks[-1]
        chunks[-2] = chunks[-2][:-needed]
    assert chunks and all(2 <= len(chunk) <= max_length for chunk in chunks)
    assert chunks[-1][-1] == eos_token_id
    assert [token for chunk in chunks for token in chunk] == sequence
    return chunks


def scan_split(folder):
    """Return eligible paths plus a strong hash of the cleaned corpus."""
    hasher = hashlib.sha256()
    paths, raw_chars, clean_chars = [], 0, 0
    for path in tqdm(sorted(folder.glob("*.md")), desc=f"scan {folder.parent.name}", unit="paper"):
        raw = path.read_text(encoding="utf-8", errors="replace")
        body = clean_paper(raw)
        if len(body) < MIN_CHARS:
            continue
        encoded_name = path.name.encode("utf-8")
        encoded_body = body.encode("utf-8")
        hasher.update(len(encoded_name).to_bytes(4, "big"))
        hasher.update(encoded_name)
        hasher.update(hashlib.sha256(encoded_body).digest())
        paths.append(path)
        raw_chars += len(raw)
        clean_chars += len(body)
    return paths, hasher.hexdigest(), raw_chars, clean_chars


train_paths, TRAIN_CORPUS_SHA256, raw_chars, clean_chars = scan_split(CORPUS)
all_eval_paths, _, _, _ = scan_split(MUFASA_ROOT / "evaluate" / "markdown")
EVAL_PAPERS = 40
eval_paths = sorted(
    all_eval_paths,
    key=lambda path: hashlib.sha256(path.name.encode("utf-8")).hexdigest(),
)[:EVAL_PAPERS]

def selected_hash(paths):
    hasher = hashlib.sha256()
    for path in paths:
        body = clean_paper(path.read_text(encoding="utf-8", errors="replace"))
        hasher.update(path.name.encode("utf-8") + b"\0")
        hasher.update(hashlib.sha256(body.encode("utf-8")).digest())
    return hasher.hexdigest()

EVAL_CORPUS_SHA256 = selected_hash(eval_paths)
TOKENIZER_FINGERPRINT = (
    f"{MODEL_ID}|{BASE_COMMIT}|{tokenizer.__class__.__name__}|{len(tokenizer)}|"
    f"{tokenizer.eos_token_id}|{max_seq_length}|{WINDOWING_VERSION}"
)

def generate_rows(paths, corpus_fingerprint, tokenizer_fingerprint):
    # The two fingerprints are deliberately arguments: Hugging Face includes
    # them in its generator cache key, invalidating stale tokenized datasets.
    del corpus_fingerprint, tokenizer_fingerprint
    for path_string in paths:
        path = Path(path_string)
        body = clean_paper(path.read_text(encoding="utf-8", errors="replace"))
        token_ids = tokenizer(body, add_special_tokens=False).input_ids
        chunks = window_token_ids(token_ids, max_seq_length, tokenizer.eos_token_id)
        for index, chunk in enumerate(chunks):
            yield {
                "input_ids": chunk,
                "length": len(chunk),
                "paper_id": path.stem,
                "is_final_window": index == len(chunks) - 1,
            }


FEATURES = Features({
    "input_ids": Sequence(Value("int32")),
    "length": Value("int32"),
    "paper_id": Value("string"),
    "is_final_window": Value("bool"),
})
DATASET_CACHE = Path("/content/mufasa_dataset_cache") if ON_COLAB_VM else CKPT_DIR / "dataset_cache"
DATASET_CACHE.mkdir(parents=True, exist_ok=True)

dataset = Dataset.from_generator(
    generate_rows,
    gen_kwargs={
        "paths": [str(path) for path in train_paths],
        "corpus_fingerprint": TRAIN_CORPUS_SHA256,
        "tokenizer_fingerprint": TOKENIZER_FINGERPRINT,
    },
    features=FEATURES,
    cache_dir=str(DATASET_CACHE),
)
eval_dataset = Dataset.from_generator(
    generate_rows,
    gen_kwargs={
        "paths": [str(path) for path in eval_paths],
        "corpus_fingerprint": EVAL_CORPUS_SHA256,
        "tokenizer_fingerprint": TOKENIZER_FINGERPRINT,
    },
    features=FEATURES,
    cache_dir=str(DATASET_CACHE),
)

def token_count(table):
    lengths = pc.list_value_length(table.data.column("input_ids"))
    return int(pc.sum(lengths).as_py())

TRAIN_WINDOW_TOKENS = token_count(dataset)
EVAL_WINDOW_TOKENS = token_count(eval_dataset)
assert TRAIN_WINDOW_TOKENS >= len(train_paths)  # includes one EOS per retained paper
assert max(dataset["length"]) <= max_seq_length
assert max(eval_dataset["length"]) <= max_seq_length

print(f"papers kept    : {len(train_paths):,}")
print(f"characters     : {raw_chars/1e6:,.0f}M raw -> {clean_chars/1e6:,.0f}M cleaned")
print(f"train windows  : {len(dataset):,}; supervised tokens: {TRAIN_WINDOW_TOKENS/1e6:,.2f}M")
print(f"held-out       : {len(eval_paths)} papers / {len(eval_dataset):,} complete windows")
print(f"corpus sha256  : {TRAIN_CORPUS_SHA256}")
print("\nfirst 400 decoded characters:\n")
print(tokenizer.decode(dataset[0]["input_ids"], skip_special_tokens=False)[:400])


<a name="Train"></a>
### Continued Pretraining
Now let's use Unsloth's `UnslothTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 20 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

Also set `embedding_learning_rate` to be a learning rate at least 2x or 10x smaller than `learning_rate` to make continual pretraining work!


In [ ]:
# ================= trainer, manifest, and held-out selection =================
import json
import os
import platform
from datetime import datetime, timezone

from trl.trainer.sft_trainer import DataCollatorForLanguageModeling
from unsloth import UnslothTrainer, UnslothTrainingArguments

LEARNING_RATE = 5e-5
EMBEDDING_LEARNING_RATE = 1e-5
QWEN_GDN_FLOAT32_GUARD = True

collator = DataCollatorForLanguageModeling(
    pad_token_id=tokenizer.pad_token_id,
    completion_only_loss=False,
    pad_to_multiple_of=8,
)

RUN_MANIFEST = {
    "schema": RUN_SCHEMA,
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "base_commit": BASE_COMMIT,
    "model_api": MODEL_API.__name__,
    "packages": PACKAGE_VERSIONS,
    "python": platform.python_version(),
    "tokenizer": {
        "class": tokenizer.__class__.__name__, "vocab_size": len(tokenizer),
        "eos_token_id": tokenizer.eos_token_id, "pad_token_id": tokenizer.pad_token_id,
    },
    "dataset": {
        "cleaner_version": CLEANER_VERSION,
        "windowing_version": WINDOWING_VERSION,
        "drop_references": DROP_REFERENCES,
        "drop_front_matter": DROP_FRONT_MATTER,
        "minimum_clean_characters": MIN_CHARS,
        "minimum_final_window_tokens": MIN_FINAL_WINDOW,
        "train_corpus_sha256": TRAIN_CORPUS_SHA256,
        "eval_corpus_sha256": EVAL_CORPUS_SHA256,
        "train_papers": len(train_paths), "train_windows": len(dataset),
        "train_tokens_including_eos": TRAIN_WINDOW_TOKENS,
        "eval_papers": len(eval_paths), "eval_windows": len(eval_dataset),
        "max_seq_length": max_seq_length,
    },
    "lora": {
        "r": LORA_R, "alpha": LORA_ALPHA, "rslora": True,
        "dropout": 0, "bias": "none", "gradient_checkpointing": "unsloth",
        "targets": TARGET_MODULES, "adapter_contract": ADAPTER_CONTRACT,
    },
    "training": {
        "epochs": 1, "batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "embedding_learning_rate": EMBEDDING_LEARNING_RATE,
        "warmup_ratio": 0.05, "weight_decay": 0.001,
        "scheduler": "linear", "optimizer": "adamw_8bit",
        "bf16": USE_BF16, "fp16": USE_FP16,
        "qwen_gdn_float32_guard": QWEN_GDN_FLOAT32_GUARD,
        "low_memory_loss": os.environ.get("UNSLOTH_RETURN_LOGITS"),
        "pretokenized_input_ids": True,
        "skip_prepare_dataset": True,
        "completion_only_loss": False, "packing": False,
        "eval_packing": False, "padding_free": False,
        "collator": "trl.DataCollatorForLanguageModeling",
        "pad_to_multiple_of": 8,
        "group_by_length": True,
        "save_steps": SAVE_STEPS, "eval_steps": SAVE_STEPS,
        "save_total_limit": 3, "load_best_model_at_end": True,
        "best_metric": "eval_loss", "greater_is_better": False,
        "preflight_steps": PREFLIGHT_STEPS, "seed": SEED,
    },
}

manifest_path = CKPT_DIR / "run_manifest.json"
checkpoint_dirs = sorted(CKPT_DIR.glob("checkpoint-*"))
if checkpoint_dirs and not manifest_path.exists():
    raise RuntimeError(
        f"found {len(checkpoint_dirs)} unverified checkpoint(s) without run_manifest.json; "
        "use a new RUN_NAME or restore the original manifest"
    )
if manifest_path.exists():
    existing = json.loads(manifest_path.read_text(encoding="utf-8"))
    if existing != RUN_MANIFEST:
        different = sorted(key for key in set(existing) | set(RUN_MANIFEST)
                           if existing.get(key) != RUN_MANIFEST.get(key))
        raise RuntimeError(
            f"checkpoint contract changed in {CKPT_DIR} (sections: {different}). "
            "Use a new RUN_NAME; never resume incompatible weights."
        )
else:
    temporary = manifest_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(RUN_MANIFEST, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temporary, manifest_path)

trainer = UnslothTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    args=UnslothTrainingArguments(
        max_length=max_seq_length,
        packing=False,
        eval_packing=False,
        padding_free=False,
        completion_only_loss=False,
        dataset_kwargs={"skip_prepare_dataset": True},
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        per_device_eval_batch_size=1,
        num_train_epochs=1,
        warmup_ratio=0.05,
        learning_rate=LEARNING_RATE,
        embedding_learning_rate=EMBEDDING_LEARNING_RATE,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        bf16=USE_BF16,
        fp16=USE_FP16,
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        group_by_length=True,
        seed=SEED,
        data_seed=SEED,
        output_dir=str(CKPT_DIR),
        report_to="none",
    ),
)
assert trainer.args.max_length == max_seq_length
assert trainer.args.packing is False and trainer.args.padding_free is False
assert trainer.args.dataset_kwargs.get("skip_prepare_dataset") is True
print("run manifest:", manifest_path)


In [ ]:
# @title Initial GPU memory
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = torch.cuda.max_memory_reserved() / 1024**3
max_memory = gpu_stats.total_memory / 1024**3
print(f"GPU = {gpu_stats.name}; capacity = {max_memory:.2f} GiB")
print(f"reserved after model+trainer = {start_gpu_memory:.2f} GiB")


In [ ]:
# ======================= real optimizer preflight ============================
# Two actual optimizer updates on full-length rows. This allocates the real
# AdamW8bit state and exercises the selected precision/loss path. A finally
# block restores every trainable tensor and RNG even when the preflight fails.
from contextlib import nullcontext
import random

import numpy as np
import torch

assert PREFLIGHT_STEPS >= 1
examples = []
for row in dataset:
    if row["length"] == max_seq_length:
        examples.append({"input_ids": row["input_ids"]})
    if len(examples) == TRAIN_BATCH_SIZE:
        break
assert examples, "no full-length window available for the memory preflight"
while len(examples) < TRAIN_BATCH_SIZE:
    examples.append(examples[-1])

trainable = [(name, param) for name, param in model.named_parameters() if param.requires_grad]
trainable_by_name = dict(trainable)
assert trainable, "no trainable parameters"
snapshot = {name: param.detach().cpu().clone() for name, param in trainable}
python_rng_state = random.getstate()
numpy_rng_state = np.random.get_state()
cpu_rng_state = torch.random.get_rng_state()
cuda_rng_states = torch.cuda.get_rng_state_all()

def target_name(parameter_name):
    for target in TARGET_MODULES:
        if f".{target}." in parameter_name:
            return target
    return None

probes = {}
for name, param in trainable:
    target = target_name(name)
    is_update_weight = (
        "lora_B" in name
        or "lora_embedding_B" in name
        or "modules_to_save.default.weight" in name
    )
    if target and target not in probes and is_update_weight:
        probes[target] = name
assert set(probes) == set(TARGET_MODULES), sorted(set(TARGET_MODULES) - set(probes))

optimizer = scaler = batch = None
losses = []
peak_allocated = peak_reserved = 0.0
try:
    trainer.create_optimizer()
    optimizer = trainer.optimizer
    batch = collator(examples)
    device = next(param.device for _, param in trainable)
    batch = {key: value.to(device) for key, value in batch.items()}

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)
    amp_dtype = torch.bfloat16 if USE_BF16 else torch.float16
    gradient_targets = set()
    model.train()

    for _ in range(PREFLIGHT_STEPS):
        optimizer.zero_grad(set_to_none=True)
        context = torch.autocast("cuda", dtype=amp_dtype) if (USE_BF16 or USE_FP16) else nullcontext()
        with context:
            loss = model(**batch).loss
        assert torch.isfinite(loss), f"non-finite preflight loss: {loss}"
        scaler.scale(loss).backward()
        if USE_FP16:
            scaler.unscale_(optimizer)
        for name, param in trainable:
            if param.grad is None:
                continue
            assert torch.isfinite(param.grad).all(), f"non-finite gradient: {name}"
            if torch.count_nonzero(param.grad):
                target = target_name(name)
                if target:
                    gradient_targets.add(target)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.detach().cpu()))

    missing_gradients = sorted(set(TARGET_MODULES) - gradient_targets)
    assert not missing_gradients, f"adapter families without gradients: {missing_gradients}"
    changed = {
        target for target, name in probes.items()
        if not torch.equal(trainable_by_name[name].detach().cpu(), snapshot[name])
    }
    assert changed == set(TARGET_MODULES), (
        f"adapter families not updated: {sorted(set(TARGET_MODULES) - changed)}"
    )

    peak_allocated = torch.cuda.max_memory_allocated() / 1024**3
    peak_reserved = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    assert peak_reserved < total * 0.98, (
        f"preflight left unsafe VRAM headroom: reserved {peak_reserved:.2f}/{total:.2f} GiB"
    )
finally:
    with torch.no_grad():
        for name, param in trainable:
            param.copy_(snapshot[name].to(device=param.device, dtype=param.dtype))
    for target, name in probes.items():
        assert torch.equal(trainable_by_name[name].detach().cpu(), snapshot[name]), (
            f"restore failed: {target}"
        )
    random.setstate(python_rng_state)
    np.random.set_state(numpy_rng_state)
    torch.random.set_rng_state(cpu_rng_state)
    torch.cuda.set_rng_state_all(cuda_rng_states)
    model.zero_grad(set_to_none=True)
    if optimizer is not None:
        optimizer.zero_grad(set_to_none=True)
    trainer.optimizer = None
    trainer.lr_scheduler = None
    del optimizer, scaler, batch, snapshot
    torch.cuda.empty_cache()

print(f"preflight passed: {PREFLIGHT_STEPS} optimizer steps; losses={losses}")
print(f"all {len(TARGET_MODULES)} adapter families had finite gradients and changed")
print(f"peak allocated/reserved: {peak_allocated:.2f}/{peak_reserved:.2f} of {total:.2f} GiB")


In [ ]:
# ========================== resumable full CPT ===============================
import json
from pathlib import Path

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(CKPT_DIR))
if last_checkpoint:
    checkpoint_path = Path(last_checkpoint)
    state_file = checkpoint_path / "trainer_state.json"
    if not state_file.is_file():
        raise RuntimeError(f"incomplete checkpoint: {last_checkpoint}")
    json.loads(state_file.read_text(encoding="utf-8"))
    weight_files = list(checkpoint_path.glob("*.safetensors")) + list(checkpoint_path.glob("*.bin"))
    if not weight_files:
        raise RuntimeError(f"checkpoint has no saved weights: {last_checkpoint}")
    print("resuming exactly:", last_checkpoint)
else:
    print("starting fresh under the validated v2 run contract")

trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)
train_loss = float(trainer_stats.metrics.get("train_loss", float("nan")))
assert train_loss == train_loss and train_loss < float("inf"), trainer_stats.metrics
TRAINING_COMPLETE = True
print("best checkpoint:", trainer.state.best_model_checkpoint)


### Durable export

The best held-out-loss adapter is saved to the model-specific versioned run directory.


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!


In [ ]:
# ======================== durable final adapter ==============================
import json
import os
from datetime import datetime, timezone

assert TRAINING_COMPLETE
FINAL_DIR = CKPT_DIR / "final_adapter"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
success_marker = FINAL_DIR / "_SUCCESS.json"
success_marker.unlink(missing_ok=True)
trainer.model.save_pretrained(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
(FINAL_DIR / "run_manifest.json").write_text(
    json.dumps(RUN_MANIFEST, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
(FINAL_DIR / "trainer_metrics.json").write_text(
    json.dumps(trainer_stats.metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
success = {
    "completed_at": datetime.now(timezone.utc).isoformat(),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "global_step": trainer.state.global_step,
    "train_loss": trainer_stats.metrics.get("train_loss"),
}
temporary = FINAL_DIR / "_SUCCESS.tmp"
temporary.write_text(json.dumps(success, indent=2, sort_keys=True) + "\n", encoding="utf-8")
os.replace(temporary, success_marker)
print("best-eval-loss adapter saved to", FINAL_DIR)

SAVE_MERGED = False
if SAVE_MERGED:
    trainer.model.save_pretrained_merged(
        str(CKPT_DIR / "merged_16bit"), tokenizer, save_method="merged_16bit"
    )


Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:


Optional loading and deployment cells follow. They are disabled by default and are
not part of the CPT or its evaluation.


You can also use Hugging Face's `AutoPeftModelForCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.


In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer

    model = AutoPeftModelForCausalLM.from_pretrained(
        str(CKPT_DIR / "final_adapter"),  # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained(str(CKPT_DIR / "final_adapter"))


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.


In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged(str(CKPT_DIR / "merged_16bit"), tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged(f"YOUR_HF_USERNAME/mufasa-{RUN_NAME}-16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged(str(CKPT_DIR / "merged_4bit"), tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged(f"YOUR_HF_USERNAME/mufasa-{RUN_NAME}-4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained(str(CKPT_DIR / "final_adapter"))
    tokenizer.save_pretrained(str(CKPT_DIR / "final_adapter"))
if False:
    model.push_to_hub(f"YOUR_HF_USERNAME/mufasa-{RUN_NAME}-lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub(f"YOUR_HF_USERNAME/mufasa-{RUN_NAME}-lora", token = "YOUR_HF_TOKEN")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.


In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf(str(CKPT_DIR / "gguf"), tokenizer,)
if False: model.push_to_hub_gguf(f"YOUR_HF_USERNAME/mufasa-{RUN_NAME}-gguf", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf(str(CKPT_DIR / "gguf"), tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf(f"YOUR_HF_USERNAME/mufasa-{RUN_NAME}-gguf", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf(str(CKPT_DIR / "gguf"), tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf(f"YOUR_HF_USERNAME/mufasa-{RUN_NAME}-gguf", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")
if False: model.push_to_hub_gguf(f"YOUR_HF_USERNAME/mufasa-{RUN_NAME}-gguf", tokenizer, quantization_method = "q5_k_m", token = "YOUR_HF_TOKEN")


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>


## Evaluating the CPT model

Four measurements, in two pairs. The first pair asks **did it work**, the second
asks **did it break anything**. You need both: a good domain number can sit on
top of a wrecked model.

| measurement | direction | what it tells you | why it matters |
|---|---|---|---|
| **domain perplexity** on held-out African papers | lower is better | how confidently the model predicts this literature | the headline result — CPT exists to move this |
| **entity NLL, trained papers** | lower is better | *retention* — did it absorb what it read | this is "knows what was studied in Sokoto" |
| **entity NLL, held-out papers** | lower is better | *generalisation* — does that help on unseen research | separates learning the domain from memorising papers |
| **general perplexity** on ordinary English | lower is better | what the domain gain cost | a large rise is catastrophic forgetting |
| **ARC / HellaSwag / PIQA / MMLU** | higher is better | general capability | catches loss the perplexity numbers miss |

**How to read the combinations**

- domain ↓, general ≈ flat → the good case; keep it
- domain ↓, general ↑↑ → paying for the domain with the model; lower the LR or train fewer steps
- retention ↓ but held-out flat → memorised the papers rather than learning the field
- nothing moved → usually `embed_tokens` / `lm_head` missing from `target_modules`, or the corpus cell never replaced the template's example data

Entity NLL is scored **only on the target span** — a place name, a species, a
figure — not on the surrounding prose, so it measures knowing the thing rather
than writing fluently around it.


In [ ]:
# ==================== evaluate the best CPT adapter ==========================
import gc
import hashlib
import importlib
from pathlib import Path

import pandas as pd

import types
MUFASA_EVAL_SHA256 = "bef5c2b89060c82cbfe2e7e0005923fbc9359e9457d6caa0e44e76ff7a7b1b27"
MUFASA_EVAL_SOURCE = '"""Measuring what continued pretraining did, against the model it started from.\n\nEverything here works on a raw completion model - no chat template, no\ninstruction following - because that is what a CPT checkpoint is.\n\nTwo things are measured, and they answer different questions.\n\n  WITHIN a model: base vs CPT\n      Perplexity is fine here. Both sides use the same tokenizer, so the ratio\n      between them is honest, and it is the headline CPT result.\n\n  ACROSS models: Gemma vs Qwen vs LFM\n      Perplexity is NOT comparable and must never be used for this. It is a\n      per-TOKEN quantity, and the three tokenizers cut the same text into\n      different numbers of tokens. Measured on this corpus:\n\n          LFM2.5   0.295 tokens/byte\n          Gemma 3  0.314 tokens/byte\n          Qwen3.5  0.324 tokens/byte\n\n      For three models of IDENTICAL quality (1.10 bits per byte) that produces\n      perplexities of 13.30, 11.37 and 10.54 - a 26% spread that is purely an\n      artifact of tokenisation, and it would rank LFM last for no reason.\n\n      Bits per byte divides the same total information by BYTES instead of\n      tokens, so the tokenizer cancels out. Use it, and only it, to compare\n      the three models with each other.\n\n  span NLL       Negative log-likelihood of a target span - a place, a technical\n                 term, a measured figure - with the surrounding prose masked out.\n                 Called span NLL and not "entity recall" because that is what\n                 it is: a likelihood, not a recall rate. Nothing is retrieved\n                 and nothing is counted as correct or incorrect. LOWER means\n                 the model finds the true continuation less surprising.\n\n                 Measured twice: on TRAIN papers (retention - did it absorb\n                 what it read) and on EVALUATE papers (generalisation - does\n                 that help on research it never saw). Report both.\n\n  standard benchmarks   ARC, HellaSwag, PIQA, MMLU via lm-evaluation-harness.\n                 HIGHER is better. These detect capability lost while the\n                 language-modelling numbers looked fine.\n"""\n\nimport math\nimport re\nfrom collections import Counter\n\nimport pandas as pd\nimport torch\n\nWINDOW = 2048          # fixed, so base and CPT are scored on identical spans\n\n\n# ------------------------------------------------- perplexity / bits-per-byte --\n\n@torch.no_grad()\ndef score_text(model, tokenizer, texts, window=WINDOW, limit=None, progress=None):\n    """Perplexity AND bits-per-byte over the same forward passes.\n\n    Returns a dict. Use `perplexity` to compare a model against its own base;\n    use `bits_per_byte` to compare different models with each other. They come\n    from one measurement because computing them separately would double the\n    cost for no reason.\n\n    Scored in fixed windows rather than whole documents, so a long paper and a\n    short one contribute in proportion to their length and the number does not\n    move when max_seq_length changes.\n    """\n    model.eval()\n    total_nll, total_tokens, total_bytes = 0.0, 0, 0\n    for index, text in enumerate(texts[:limit] if limit else texts):\n        ids = tokenizer(text, return_tensors="pt").input_ids[0]\n        scored_to = 0\n        for start in range(0, max(len(ids) - 1, 0), window):\n            chunk = ids[start:start + window]\n            if len(chunk) < 32:                      # too short to score fairly\n                continue\n            chunk = chunk.unsqueeze(0).to(model.device)\n            out = model(chunk, labels=chunk)\n            counted = chunk.numel() - 1\n            total_nll += out.loss.item() * counted\n            total_tokens += counted\n            scored_to = start + len(chunk[0])\n        # Only the bytes actually scored count, or a document skipped for being\n        # short would still inflate the denominator.\n        if scored_to:\n            decoded = tokenizer.decode(ids[:scored_to], skip_special_tokens=True)\n            total_bytes += len(decoded.encode("utf-8"))\n        if progress:\n            progress(index + 1)\n    if not total_tokens or not total_bytes:\n        return {"perplexity": float("nan"), "bits_per_byte": float("nan"),\n                "tokens": 0, "bytes": 0}\n    return {\n        "perplexity": math.exp(total_nll / total_tokens),\n        "bits_per_byte": total_nll / math.log(2) / total_bytes,\n        "tokens": total_tokens,\n        "bytes": total_bytes,\n    }\n\n\ndef perplexity(model, tokenizer, texts, window=WINDOW, limit=None, progress=None):\n    """Back-compatible wrapper returning (perplexity, tokens)."""\n    got = score_text(model, tokenizer, texts, window, limit, progress)\n    return got["perplexity"], got["tokens"]\n\n\n# ------------------------------------------------------------- span targets --\n\nPLACE = re.compile(r"\\b(?:in|at|from|across)\\s+([A-Z][a-z]+(?:\\s+[A-Z][a-z]+){0,2})")\n# Not a taxonomy matcher. This is a capitalised technical bigram - "Atterberg\n# limits", "Uyo metropolis", sometimes a real binomial. Calling it SPECIES would\n# be a lie: on this corpus most matches are domain terms, not organisms. What\n# makes it usable as a probe is the recurrence test in _recurring(), which drops\n# one-off sentence fragments like "Studying land".\nTERM = re.compile(r"\\b([A-Z][a-z]+\\s+[a-z]{4,})\\b")\nSPECIES = TERM      # kept so older references still resolve\nFIGURE = re.compile(r"\\b(\\d+\\.\\d+)\\s*(?:%|mg|kg|ml|cfu|ppm|ppb|mm|cm|km|ha|mS|C\\b)")\n\n# Words that are page furniture, not knowledge. A model predicting "Table" after\n# "shown in" has demonstrated nothing about African research.\nSTRUCTURAL = {\n    "Fig", "Figure", "Figures", "Table", "Tables", "Section", "Appendix", "Plate",\n    "Chart", "Map", "Photo", "Equation", "Eq", "Source", "Note", "Total", "Chapter",\n    "Page", "Annex", "Box", "Panel", "Sci", "Vol", "No", "Ref", "Abstract",\n}\nMONTHS = {"January", "February", "March", "April", "May", "June", "July", "August",\n          "September", "October", "November", "December"}\n# Sentence scaffolding that the capitalisation heuristic picks up by accident.\nGENERIC = {\n    "The", "This", "These", "Those", "There", "However", "Although", "Therefore",\n    "Parts", "Part", "Results", "Result", "Discussion", "Introduction", "Methods",\n    "Conclusion", "Study", "Data", "Analysis", "Modified", "Based", "Using", "Both",\n    "Each", "Other", "Some", "Such", "Their", "Its", "It", "We", "Our", "All",\n}\nREJECT = STRUCTURAL | MONTHS | GENERIC\n\n\ndef _worth_scoring(target):\n    """Is this span actually carrying knowledge?"""\n    words = target.split()\n    if not words or len(target) < 4:\n        return False\n    if any(word in REJECT for word in words):\n        return False\n    return True\n\n\ndef _recurring(texts, minimum=2):\n    """Candidate terms that appear more than once across the sampled papers.\n\n    A term used twice is part of the literature\'s vocabulary; one used once is\n    usually a sentence fragment the capitalisation rule caught by accident.\n    """\n    seen = Counter()\n    for text in texts:\n        for match in TERM.finditer(text):\n            seen[match.group(1)] += 1\n    return {term for term, n in seen.items() if n >= minimum}\n\n\ndef entity_items(texts, per_pattern=2, limit=400, max_share=0.04):\n    """(context, target) pairs where the target is a place, term or figure.\n\n    Four things this gets right that a single quota loop does not.\n\n    Each pattern has its OWN quota. With one shared quota the first pattern\n    fills it every time: measured on 40 papers, PLACE produced 240 of 240\n    targets and terms and figures were never sampled at all.\n\n    Terms must RECUR across the sampled papers. A capitalised bigram used once\n    is usually a sentence fragment ("Studying land"); one used twice is part of\n    the literature\'s vocabulary ("Atterberg limits").\n\n    Page furniture is rejected. "Fig", "Table", "November", "Modified" all match\n    a capitalised-word-after-preposition rule and none of them test knowledge.\n\n    No single target may exceed `max_share` of the set. Unfiltered, "Nigeria"\n    was 60 of 240 targets - a quarter of the measurement spent on the one word\n    every paper in the corpus contains.\n    """\n    items, used = [], Counter()\n    cap = max(1, int(limit * max_share))\n    recurring = _recurring(texts)\n    for text in texts:\n        for pattern in (PLACE, TERM, FIGURE):\n            found = 0\n            for match in pattern.finditer(text):\n                if found >= per_pattern or len(items) >= limit:\n                    break\n                start = match.start(1)\n                target = match.group(1)\n                if start < 200:                      # need real context first\n                    continue\n                if not _worth_scoring(target) or used[target] >= cap:\n                    continue\n                if pattern is TERM and target not in recurring:\n                    continue\n                items.append((text[max(0, start - 600):start], target))\n                used[target] += 1\n                found += 1\n        if len(items) >= limit:\n            break\n    return items[:limit]\n\n\n@torch.no_grad()\ndef entity_nll(model, tokenizer, items, progress=None):\n    """Mean NLL of the target spans, with the context masked out of the loss."""\n    model.eval()\n    total, counted = 0.0, 0\n    for index, (context, target) in enumerate(items):\n        context_ids = tokenizer(context, return_tensors="pt").input_ids[0][-1024:]\n        target_ids = tokenizer(target, add_special_tokens=False,\n                               return_tensors="pt").input_ids[0]\n        if not len(target_ids):\n            continue\n        ids = torch.cat([context_ids, target_ids]).unsqueeze(0).to(model.device)\n        labels = ids.clone()\n        labels[0, :len(context_ids)] = -100          # score the target only\n        out = model(ids, labels=labels)\n        total += out.loss.item() * len(target_ids)\n        counted += len(target_ids)\n        if progress:\n            progress(index + 1)\n    return (total / counted) if counted else float("nan"), counted\n\n\ndef describe_items(items):\n    """What the probe is actually made of - check this before trusting a score."""\n    targets = [t for _, t in items]\n    kinds = Counter()\n    for t in targets:\n        if re.fullmatch(r"\\d+\\.\\d+", t):\n            kinds["figure"] += 1\n        elif TERM.fullmatch(t):\n            kinds["term"] += 1\n        else:\n            kinds["place"] += 1\n    common = Counter(targets).most_common(1)\n    return {"targets": len(targets), "distinct": len(set(targets)),\n            "by kind": dict(kinds),\n            "most common": f"{common[0][0]} x{common[0][1]}" if common else "-",\n            "top share": f"{100 * common[0][1] / len(targets):.0f}%" if common else "-"}\n\n\n# --------------------------------------------------------------- reporting --\n\nLOWER_IS_BETTER = {\n    "domain perplexity": True, "general perplexity": True,\n    "domain bits/byte": True, "general bits/byte": True,\n    "span NLL (train papers)": True, "span NLL (held out)": True,\n}\n\n\ndef compare(base, tuned, base_name="base", tuned_name="CPT"):\n    """One table. `LOWER_IS_BETTER` decides which direction counts as a win."""\n    rows = []\n    for metric in base:\n        before, after = base[metric], tuned[metric]\n        lower = LOWER_IS_BETTER.get(metric, True)\n        change = (after - before) / before * 100 if before else float("nan")\n        improved = (after < before) if lower else (after > before)\n        rows.append({"metric": metric, base_name: round(before, 4),\n                     tuned_name: round(after, 4),\n                     "change %": round(change, 1),\n                     "better?": "yes" if improved else "no"})\n    return pd.DataFrame(rows)\n\n\ndef read_out(frame, base_name="base", tuned_name="CPT"):\n    """Say in words what the table means, including the awkward combinations."""\n    def get(metric, column):\n        row = frame[frame.metric == metric]\n        return float(row[column].iloc[0]) if len(row) else float("nan")\n\n    lines = []\n    domain = get("domain perplexity", "change %")\n    general = get("general perplexity", "change %")\n    retention = get("span NLL (train papers)", "change %")\n    held = get("span NLL (held out)", "change %")\n\n    if domain < -5:\n        lines.append(f"Domain perplexity fell {abs(domain):.0f}% - the model predicts "\n                     "African research writing markedly better than the base did. "\n                     "This is the result CPT exists for.")\n    elif domain < 0:\n        lines.append(f"Domain perplexity fell only {abs(domain):.0f}%. Real but small: "\n                     "more epochs, a higher rank, or more tokens would be the levers.")\n    else:\n        lines.append(f"Domain perplexity ROSE {domain:.0f}%. Something is wrong - "\n                     "check the learning rate, and that the corpus cell is feeding "\n                     "papers rather than the template\'s example dataset.")\n\n    if general > 25:\n        lines.append(f"General perplexity rose {general:.0f}%. That is catastrophic "\n                     "forgetting: the model is paying for the domain with its general "\n                     "ability. Lower the learning rate or train fewer steps.")\n    elif general > 5:\n        lines.append(f"General perplexity rose {general:.0f}% - the ordinary price of "\n                     "domain adaptation, and the SFT stage will recover some of it.")\n    else:\n        lines.append(f"General perplexity moved {general:+.0f}% - essentially intact, "\n                     "so nothing was traded away for the domain gain.")\n\n    if retention < -5 and held < -5:\n        lines.append("Span NLL improved on BOTH trained and held-out papers, so the "\n                     "model learned the vocabulary of this literature, not just the "\n                     "specific papers it read.")\n    elif retention < -5 <= held:\n        lines.append("Span NLL improved on trained papers but not held-out ones: it "\n                     "memorised rather than generalised. Fine if you only need recall "\n                     "of the corpus, weak if you want it to handle new research.")\n    elif retention >= -5:\n        lines.append("Span NLL barely moved even on trained papers. The spans are not "\n                     "being absorbed - the usual cause is embed_tokens and lm_head "\n                     "missing from target_modules.")\n\n    lines.append("To compare this model with the other two, use bits/byte and NOT "\n                 "perplexity: perplexity is per-token and the three tokenizers cut "\n                 "the same text differently, which alone moves it by ~26%.")\n    return lines\n\n\ndef plot(frame, base_name="base", tuned_name="CPT", title="CPT vs base"):\n    """Grouped bars per metric, plus the percentage change beside them.\n\n    Perplexity and bits-per-byte live on different scales, so the raw-value\n    panel is split - drawing them on one axis would flatten bits/byte to\n    nothing against a perplexity of 12.\n    """\n    import matplotlib.pyplot as plt\n    import numpy as np\n\n    ppl = frame[~frame.metric.str.contains("bits/byte")].reset_index(drop=True)\n    bpb = frame[frame.metric.str.contains("bits/byte")].reset_index(drop=True)\n    panels = 3 if len(bpb) else 2\n    widths = [3, 1.3, 2] if panels == 3 else [3, 2]\n    figure, axes = plt.subplots(1, panels, figsize=(4.4 * panels, 4.2),\n                                gridspec_kw={"width_ratios": widths})\n\n    def bars(ax, data, heading):\n        x = np.arange(len(data))\n        width = 0.38\n        ax.bar(x - width / 2, data[base_name], width, label=base_name, color="#9aa4b2")\n        ax.bar(x + width / 2, data[tuned_name], width, label=tuned_name, color="#2f5d8c")\n        ax.set_xticks(x)\n        ax.set_xticklabels([m.replace(" (", "\\n(").replace(" perplexity", "\\nperplexity")\n                            for m in data.metric], fontsize=8)\n        ax.set_title(heading, fontsize=10)\n        ax.legend(fontsize=8)\n        ax.grid(axis="y", alpha=0.25)\n\n    bars(axes[0], ppl, f"{title} - lower is better")\n    if panels == 3:\n        bars(axes[1], bpb, "bits/byte\\n(cross-model)")\n\n    change = frame["change %"]\n    axes[-1].barh(frame.metric, change,\n                  color=["#3e7a5e" if c < 0 else "#a34434" for c in change])\n    axes[-1].axvline(0, color="#333", linewidth=0.8)\n    axes[-1].set_title("change % (green = improved)", fontsize=10)\n    axes[-1].tick_params(labelsize=8)\n    axes[-1].grid(axis="x", alpha=0.25)\n    plt.tight_layout()\n    return figure\n\n\ndef across_models(csv_paths):\n    """Rank the three models honestly, on bits per byte only.\n\n    Each path is a cpt_evaluation.csv written by one model\'s notebook.\n    """\n    rows = []\n    for path in csv_paths:\n        frame = pd.read_csv(path)\n        got = frame[frame.metric == "domain bits/byte"]\n        if not len(got):\n            continue\n        rows.append({"model": str(path).replace("cpt_evaluation_", "").replace(".csv", ""),\n                     "base bits/byte": got["base"].iloc[0],\n                     "CPT bits/byte": got["CPT"].iloc[0],\n                     "change %": got["change %"].iloc[0]})\n    table = pd.DataFrame(rows).sort_values("CPT bits/byte")\n    return table.reset_index(drop=True)\n'
assert hashlib.sha256(MUFASA_EVAL_SOURCE.encode("utf-8")).hexdigest() == MUFASA_EVAL_SHA256
ev = types.ModuleType("mufasa_eval_embedded")
exec(compile(MUFASA_EVAL_SOURCE, "<mufasa_eval_embedded>", "exec"), ev.__dict__)
required_eval_api = {"score_text", "entity_items", "entity_nll", "describe_items", "compare", "plot", "read_out"}
assert required_eval_api <= set(vars(ev)), required_eval_api - set(vars(ev))

EVAL_DIR = MUFASA_ROOT / "evaluate" / "markdown"
TRAIN_DIR = MUFASA_ROOT / "train" / "markdown"
N_PAPERS = 40
N_ENTITIES = 300

def load_texts(folder, count):
    """Same cleaner as CPT; stable hash sample instead of alphabetical bias."""
    paths = sorted(
        Path(folder).glob("*.md"),
        key=lambda path: hashlib.sha256(path.name.encode("utf-8")).hexdigest(),
    )
    texts = []
    for path in paths:
        body = clean_paper(path.read_text(encoding="utf-8", errors="replace"))
        if len(body) >= MIN_CHARS:
            texts.append(body)
        if len(texts) == count:
            break
    return texts

held_out = load_texts(EVAL_DIR, N_PAPERS)
trained_on = load_texts(TRAIN_DIR, N_PAPERS)
print(f"held-out papers {len(held_out)}; trained-on papers {len(trained_on)}")

from datasets import load_dataset
general_rows = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")["text"]
general = ["\n".join(text for text in general_rows if len(text) > 200)[:400_000]]

held_items = ev.entity_items(held_out)
trained_items = ev.entity_items(trained_on)
print("held-out span probe:", ev.describe_items(held_items))
print("trained span probe :", ev.describe_items(trained_items))

def measure(candidate, candidate_tokenizer, label):
    from tqdm.auto import tqdm
    scores = {}
    bar = tqdm(total=4, desc=f"measuring {label}", unit="metric")
    domain = ev.score_text(candidate, candidate_tokenizer, held_out); bar.update(1)
    ordinary = ev.score_text(candidate, candidate_tokenizer, general); bar.update(1)
    scores["domain perplexity"] = domain["perplexity"]
    scores["domain bits/byte"] = domain["bits_per_byte"]
    scores["general perplexity"] = ordinary["perplexity"]
    scores["general bits/byte"] = ordinary["bits_per_byte"]
    scores["span NLL (train papers)"], _ = ev.entity_nll(
        candidate, candidate_tokenizer, trained_items[:N_ENTITIES]
    ); bar.update(1)
    scores["span NLL (held out)"], _ = ev.entity_nll(
        candidate, candidate_tokenizer, held_items[:N_ENTITIES]
    ); bar.update(1)
    bar.close()
    for name, value in scores.items():
        print(f"   {name:<28} {value:>9.4f}")
    return scores

cpt_model = trainer.model
MODEL_API.for_inference(cpt_model)
cpt_scores = dict(measure(cpt_model, tokenizer, "CPT model"))


In [ ]:
# ============= African research concept recall/association probe ==========
# This is a deterministic paper-prefix cloze probe, not a free-form QA test.
# It compares the SAME model with CPT enabled and disabled. Positive raw gain
# means better prediction of the true African concept; positive association
# gain means the paper context helped beyond merely learning the concept name.
import hashlib
import json
import math
import random
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

RECALL_SEED = 20260824
RECALL_PAPERS = 200            # deterministic papers loaded per split
RECALL_PER_SPLIT = 200         # requested probes per split after filtering
MAX_PER_CONCEPT = 3           # prevents Nigeria or one term dominating
MAX_PER_PAPER = 2
MAX_PREFIX_TOKENS = 512
MAX_TARGET_TOKENS = 12
PREFIX_CHARS = 3_000
NEUTRAL_PREFIX = 'This African research paper discusses a place or local concept:'

AFRICAN_COUNTRIES = '''Algeria|Angola|Benin|Botswana|Burkina Faso|Burundi|Cabo Verde|Cameroon|Central African Republic|Chad|Comoros|Democratic Republic of the Congo|Republic of the Congo|Djibouti|Egypt|Equatorial Guinea|Eritrea|Eswatini|Ethiopia|Gabon|Gambia|Ghana|Guinea|Guinea-Bissau|Kenya|Lesotho|Liberia|Libya|Madagascar|Malawi|Mali|Mauritania|Mauritius|Morocco|Mozambique|Namibia|Niger|Nigeria|Rwanda|Senegal|Seychelles|Sierra Leone|Somalia|South Africa|South Sudan|Sudan|Tanzania|Togo|Tunisia|Uganda|Zambia|Zimbabwe'''.split('|')
NIGERIAN_PLACES = '''Abia|Adamawa|Akwa Ibom|Anambra|Bauchi|Bayelsa|Benue|Borno|Cross River|Delta|Ebonyi|Edo|Ekiti|Enugu|Gombe|Imo|Jigawa|Kaduna|Kano|Katsina|Kebbi|Kogi|Kwara|Lagos|Nasarawa|Niger State|Ogun|Ondo|Osun|Oyo|Plateau|Rivers|Sokoto|Taraba|Yobe|Zamfara|Abuja|Federal Capital Territory|Ibadan|Ilorin|Maiduguri|Minna|Zaria|Akure|Abeokuta|Calabar|Uyo|Owerri|Port Harcourt|Benin City|Makurdi|Yola|Jos|Niger Delta|Lake Chad|Sahel|Congo Basin|Lake Victoria|Horn of Africa'''.split('|')
LOCAL_CONCEPTS = {
    'onugbu / bitter leaf': ('onugbu', 'onubu', 'bitter leaf', 'Vernonia amygdalina'),
    'zobo / roselle': ('zobo', 'zobo leaf', 'zobo leaves', 'zobo drink', 'roselle', 'Hibiscus sabdariffa'),
    'ugwu / fluted pumpkin': ('ugwu', 'ugu', 'fluted pumpkin', 'Telfairia occidentalis'),
    'ogbono / bush mango': ('ogbono', 'bush mango', 'Irvingia gabonensis'),
    'egusi': ('egusi', 'egusi melon', 'melon seed'),
    'iru / dawadawa': ('iru', 'dawadawa', 'African locust bean', 'Parkia biglobosa'),
    'garri': ('garri', 'gari'),
    'ogi / akamu': ('ogi', 'akamu'),
    'kunu': ('kunu', 'kunu-zaki', 'kunu zaki'),
    'tuwo shinkafa': ('tuwo shinkafa',),
    'amala': ('amala',),
    'eba': ('eba',),
    'fufu': ('fufu',),
    'suya': ('suya',),
    'kilishi': ('kilishi',),
    'shea': ('shea butter', 'shea nut', 'Vitellaria paradoxa'),
    'baobab': ('baobab', 'Adansonia digitata'),
    'Bambara groundnut': ('Bambara groundnut', 'Vigna subterranea'),
    'African yam bean': ('African yam bean', 'Sphenostylis stenocarpa'),
    'scent leaf': ('scent leaf', 'Ocimum gratissimum', 'efirin', 'nchuanwu'),
    'uziza': ('uziza', 'Piper guineense'),
    'utazi': ('utazi', 'Gongronema latifolium'),
    'uda': ('uda', 'Xylopia aethiopica'),
    'fonio': ('fonio',),
    'teff': ('teff',),
    'injera': ('injera',),
}

def _norm(value):
    value = unicodedata.normalize('NFKC', str(value)).casefold()
    return re.sub(r'\s+', ' ', value).strip()

concepts = []
for place in AFRICAN_COUNTRIES:
    aliases = (place, 'Nigerian') if place == 'Nigeria' else (place,)
    concepts.append(('place', place, aliases))
for place in NIGERIAN_PLACES:
    aliases = (place, place + ' State') if not place.endswith(('State', 'Territory', 'Delta', 'Chad', 'Sahel', 'Basin', 'Victoria', 'Africa')) else (place,)
    concepts.append(('place', place, aliases))
for canonical, aliases in LOCAL_CONCEPTS.items():
    concepts.append(('local_concept', canonical, aliases))

alias_lookup = {}
concept_aliases = {}
for category, canonical, aliases in concepts:
    concept_aliases[(category, canonical)] = tuple(aliases)
    for alias in aliases:
        alias_lookup.setdefault(_norm(alias), (category, canonical))
alias_terms = sorted(alias_lookup, key=lambda value: (-len(value), value))
alias_pattern = re.compile(
    r'(?<![\w])(?:' + '|'.join(re.escape(term).replace(r'\ ', r'\s+') for term in alias_terms) + r')(?![\w])',
    re.IGNORECASE,
)

def _contains_alias(prompt, aliases):
    normalized = _norm(prompt)
    return any(re.search(r'(?<!\w)' + re.escape(_norm(alias)) + r'(?!\w)', normalized) for alias in aliases)

def _paper_candidates(text, split_name, paper_number):
    paper_id = hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]
    seen = set()
    rows = []
    for match in alias_pattern.finditer(text):
        record = alias_lookup.get(_norm(match.group(0)))
        if record is None or match.start() < 200 or not text[match.start() - 1].isspace():
            continue
        category, canonical = record
        surface = text[match.start():match.end()]
        if category == 'place' and not surface[:1].isupper():
            continue
        if category == 'local_concept' and len(_norm(surface)) <= 4 and surface != surface.lower():
            continue
        key = (paper_id, category, canonical)
        if key in seen:
            continue
        prompt = text[max(0, match.start() - PREFIX_CHARS):match.start()].rstrip()
        aliases = concept_aliases[(category, canonical)]
        if len(prompt) < 200 or _contains_alias(prompt, aliases):
            continue
        seen.add(key)
        rows.append({
            'split': split_name, 'paper_id': paper_id, 'paper_number': paper_number,
            'category': category, 'canonical': canonical,
            'target': surface, 'prompt': prompt,
            'source_offset': match.start(),
        })
    return rows

def _sample_candidates(texts, split_name, seed):
    pool = []
    for paper_number, text in enumerate(texts):
        pool.extend(_paper_candidates(text, split_name, paper_number))
    pool.sort(key=lambda row: (row['paper_id'], row['source_offset'], row['canonical'], row['target']))
    random.Random(seed).shuffle(pool)
    selected, per_concept, per_paper = [], Counter(), Counter()
    for desired_category in ('local_concept', 'place', None):
        category_limit = RECALL_PER_SPLIT // 2 if desired_category else RECALL_PER_SPLIT
        for row in pool:
            if row in selected or (desired_category and row['category'] != desired_category):
                continue
            if desired_category and sum(item['category'] == desired_category for item in selected) >= category_limit:
                break
            if per_concept[row['canonical']] >= MAX_PER_CONCEPT or per_paper[row['paper_id']] >= MAX_PER_PAPER:
                continue
            selected.append(row)
            per_concept[row['canonical']] += 1
            per_paper[row['paper_id']] += 1
            if len(selected) >= RECALL_PER_SPLIT:
                return selected
    return selected

# Reuse the model from the matched adapter-toggle sanity cell or the live trainer.
recall_model = None
for candidate_name in ('cpt', 'paired_model', 'cpt_model'):
    candidate = globals().get(candidate_name)
    if candidate is not None and hasattr(candidate, 'disable_adapter'):
        recall_model = candidate
        break
if recall_model is None:
    raise RuntimeError('Run the matched adapter-toggle sanity cell first, or run this cell before the notebook frees trainer.model.')
recall_tokenizer = globals().get('tokenizer') or globals().get('eval_tokenizer') or globals().get('base_tokenizer')
if recall_tokenizer is None or not getattr(recall_tokenizer, 'is_fast', False):
    raise RuntimeError('This probe requires the matching fast tokenizer with offset mappings.')
MODEL_API.for_inference(recall_model)

recall_trained_on = load_texts(TRAIN_DIR, RECALL_PAPERS)
recall_held_out = load_texts(EVAL_DIR, RECALL_PAPERS)
print('recall papers:', len(recall_trained_on), 'trained;', len(recall_held_out), 'held out')
train_hashes = {hashlib.sha256(text.encode('utf-8')).hexdigest() for text in recall_trained_on}
held_hashes = {hashlib.sha256(text.encode('utf-8')).hexdigest() for text in recall_held_out}
assert train_hashes.isdisjoint(held_hashes), 'trained and held-out probe papers overlap'
sample = _sample_candidates(recall_trained_on, 'trained_papers', RECALL_SEED)
sample += _sample_candidates(recall_held_out, 'held_out_papers', RECALL_SEED + 1)
if not sample:
    raise RuntimeError('No African concept candidates survived the strict no-copy filters.')

def _encode_cloze(prompt, target):
    prompt = prompt.rstrip()
    combined = prompt + ' ' + target
    boundary = len(prompt)
    encoded = recall_tokenizer(combined, add_special_tokens=False, return_offsets_mapping=True)
    ids, offsets = encoded['input_ids'], encoded['offset_mapping']
    if any(start < boundary < end for start, end in offsets):
        return None
    target_positions = [index for index, (start, end) in enumerate(offsets) if start >= boundary and end > boundary]
    if not target_positions:
        return None
    first = target_positions[0]
    prefix_ids, target_ids = ids[:first], ids[first:]
    if not prefix_ids or not target_ids or len(target_ids) > MAX_TARGET_TOKENS:
        return None
    prefix_ids = prefix_ids[-MAX_PREFIX_TOKENS:]
    if recall_tokenizer.bos_token_id is not None:
        prefix_ids = [recall_tokenizer.bos_token_id] + prefix_ids
    return {'input_ids': prefix_ids + target_ids, 'prefix_len': len(prefix_ids), 'target_ids': target_ids}

items = []
for row in sample:
    conditional = _encode_cloze(row['prompt'], row['target'])
    neutral = _encode_cloze(NEUTRAL_PREFIX, row['target'])
    if conditional is None or neutral is None or conditional['target_ids'] != neutral['target_ids']:
        continue
    row = dict(row)
    row['conditional'], row['neutral'] = conditional, neutral
    items.append(row)
if not items:
    raise RuntimeError('All candidates failed exact joint-tokenization checks.')
sample_fingerprint = hashlib.sha256(json.dumps([
    (row['split'], row['paper_id'], row['source_offset'], row['canonical'], row['target'])
    for row in items
], ensure_ascii=False, separators=(',', ':')).encode('utf-8')).hexdigest()
print('recall sample:', len(items), 'items;', Counter(row['category'] for row in items))
print('sample SHA256:', sample_fingerprint)

@torch.inference_mode()
def _score_one(model, sequence):
    device = model.get_input_embeddings().weight.device
    ids = torch.tensor(sequence['input_ids'], dtype=torch.long, device=device)
    logits = model(input_ids=ids.unsqueeze(0)).logits[0]
    prefix_len = sequence['prefix_len']
    positions = torch.arange(prefix_len - 1, ids.numel() - 1, device=device)
    targets = ids[prefix_len:]
    selected_logits = logits[positions].float()
    nll = F.cross_entropy(selected_logits, targets, reduction='mean').item()
    first_logits = selected_logits[0]
    first_target = targets[0]
    top5 = torch.topk(first_logits, k=min(5, first_logits.numel())).indices
    return {
        'nll': nll,
        'first_top1': bool(first_logits.argmax() == first_target),
        'first_top5': bool((top5 == first_target).any()),
        'first_rank': int((first_logits > first_logits[first_target]).sum().item() + 1),
    }

def _score_items(model, rows, label):
    results = {}
    for index, row in enumerate(tqdm(rows, desc=label, unit='cloze')):
        results[(index, 'conditional')] = _score_one(model, row['conditional'])
        results[(index, 'neutral')] = _score_one(model, row['neutral'])
    return results

cpt_probe = _score_items(recall_model, items, 'CPT recall probe')
with recall_model.disable_adapter():
    base_probe = _score_items(recall_model, items, 'base recall probe')
# A short on/off/on reproducibility check without repeating the entire run.
recheck = _score_items(recall_model, items[:3], 'CPT state recheck')
for key, value in recheck.items():
    assert abs(value['nll'] - cpt_probe[key]['nll']) <= 1e-4, (key, value, cpt_probe[key])

records = []
for index, row in enumerate(items):
    bc, cc = base_probe[(index, 'conditional')], cpt_probe[(index, 'conditional')]
    bn, cn = base_probe[(index, 'neutral')], cpt_probe[(index, 'neutral')]
    raw_gain = bc['nll'] - cc['nll']
    neutral_gain = bn['nll'] - cn['nll']
    records.append({
        'split': row['split'], 'paper_id': row['paper_id'], 'category': row['category'],
        'canonical': row['canonical'], 'target': row['target'],
        'target_tokens': len(row['conditional']['target_ids']),
        'base_conditional_nll': bc['nll'], 'cpt_conditional_nll': cc['nll'],
        'raw_nll_gain': raw_gain, 'neutral_nll_gain': neutral_gain,
        'association_gain': raw_gain - neutral_gain,
        'base_first_top1': bc['first_top1'], 'cpt_first_top1': cc['first_top1'],
        'base_first_top5': bc['first_top5'], 'cpt_first_top5': cc['first_top5'],
        'base_first_rank': bc['first_rank'], 'cpt_first_rank': cc['first_rank'],
        'prompt_tail': row['prompt'][-180:].replace('\n', ' '),
    })
recall_frame = pd.DataFrame(records)

summary_rows = []
for split_name, group in recall_frame.groupby('split', sort=False):
    paper_gains = group.groupby('paper_id')['association_gain'].mean().to_numpy()
    rng = np.random.default_rng(RECALL_SEED)
    if len(paper_gains) > 1:
        boot = np.array([rng.choice(paper_gains, len(paper_gains), replace=True).mean() for _ in range(2_000)])
        ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
    else:
        ci_low = ci_high = paper_gains.mean()
    summary_rows.append({
        'split': split_name, 'items': len(group), 'papers': group.paper_id.nunique(),
        'distinct_concepts': group.canonical.nunique(),
        'base_span_ppl': math.exp(group.base_conditional_nll.mean()),
        'cpt_span_ppl': math.exp(group.cpt_conditional_nll.mean()),
        'conditional_win_rate': (group.raw_nll_gain > 0).mean(),
        'base_first_token_recall@5': group.base_first_top5.mean(),
        'cpt_first_token_recall@5': group.cpt_first_top5.mean(),
        'mean_raw_nll_gain': group.raw_nll_gain.mean(),
        'mean_neutral_nll_gain': group.neutral_nll_gain.mean(),
        'mean_association_gain': group.association_gain.mean(),
        'association_gain_ci95_low': ci_low, 'association_gain_ci95_high': ci_high,
    })
recall_summary = pd.DataFrame(summary_rows)
display(recall_summary.round(4))
display(recall_frame[['split', 'category', 'target', 'raw_nll_gain', 'association_gain', 'base_first_rank', 'cpt_first_rank', 'prompt_tail']].head(12))

output_path = FINAL_DIR / 'african_concept_recall_probe.csv'
recall_frame.to_csv(output_path, index=False)
print('saved:', output_path)
print('Interpretation: positive raw gain = better true-target prediction; positive association gain')
print('= paper-context learning beyond a general African-name prior. Trained-paper results test')
print('retention; held-out results test transfer. This is not yet free-form factual QA recall.')


In [ ]:
# ================= untouched-base comparison, memory-safe ====================
import gc

BASE_MODEL = None
base_id = BASE_MODEL or MODEL_ID

try:
    trainer.accelerator.free_memory()
except Exception:
    pass
try:
    if trainer.optimizer is not None:
        trainer.optimizer.zero_grad(set_to_none=True)
except Exception:
    pass

del cpt_model
del model
del trainer
del dataset
del eval_dataset
gc.collect()
torch.cuda.empty_cache()

from unsloth import FastModel
BASE_API = FastModel
base_model, base_tokenizer = BASE_API.from_pretrained(
    model_name=base_id,
    revision=BASE_COMMIT,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=False,
    text_only=True,

)
BASE_API.for_inference(base_model)
base_scores = dict(measure(base_model, base_tokenizer, "base model"))


In [ ]:
# ========================= compare, chart, persist ===========================
table = ev.compare(base_scores, cpt_scores)
display(table)
display(ev.plot(table, title=f"{base_id} — CPT vs base"))

print("\nInterpretation\n" + "=" * 72)
for line in ev.read_out(table):
    import textwrap
    print(textwrap.fill(line, 88, initial_indent="  - ", subsequent_indent="    "))

csv_name = f"cpt_evaluation_{RUN_NAME}.csv"
table.to_csv(csv_name, index=False)
table.to_csv(CKPT_DIR / csv_name, index=False)
print("\nsaved", CKPT_DIR / csv_name)


In [ ]:
# ============ standard benchmarks, via lm-evaluation-harness =================
# The four measurements above are language-modelling ones. These are the
# field-standard capability tests, and they catch losses perplexity does not.
# HIGHER is better. Run on both models and compare.
#
# Roughly 20-40 minutes for a 1B on the selected GPU; drop --limit for the full sets.
RUN_HARNESS = False

if RUN_HARNESS:
    !pip install -q lm-eval
    # general capability
    !lm_eval --model hf --model_args pretrained={base_id} \
        --tasks arc_easy,arc_challenge,hellaswag,piqa,truthfulqa_mc2 \
        --device cuda:0 --batch_size 8 --limit 500 --output_path eval_base
    # and the domains this corpus actually covers
    !lm_eval --model hf --model_args pretrained={base_id} \
        --tasks mmlu_clinical_knowledge,mmlu_college_biology,mmlu_professional_medicine \
        --device cuda:0 --batch_size 8 --limit 500 --output_path eval_base_domain
    print("now re-run both against the saved CPT model to compare")
